# Day 051 — Exercise 4: Display Helpers

**What you'll build:** `format_transcript(messages)` for an `st.download_button`, and `chat_stats(messages)` for `st.metric` widgets.

**Why it matters:** A good app doesn't just show the chat — it lets you export it and see it at a glance. These are pure functions that turn the message list into *display artifacts*: a downloadable transcript and a set of live counters. Keeping them out of the UI code means you can test them here, with no Streamlit runtime required.

## Provided: Setup + Session State + Input + Model Wiring

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import ollama


def init_session(state: dict) -> dict:
    """
    Idempotently initialise a Streamlit-style session_state dict.

    Streamlit reruns the WHOLE script top-to-bottom on every interaction, so
    initialisation must never overwrite existing data. Only set a key if absent.

    Ensures keys:
        'messages'  -> list of {'role', 'content'} dicts (starts empty)
        'settings'  -> {'model', 'temperature', 'system_prompt'}
    Returns the same dict, mutated in place.
    """
    if 'messages' not in state:
        state['messages'] = []
    if 'settings' not in state:
        state['settings'] = {
            'model': 'llama3.2',
            'temperature': 0.7,
            'system_prompt': 'You are a helpful assistant.',
        }
    return state


def add_message(state: dict, role: str, content: str) -> dict:
    """Append a {'role', 'content'} message to state['messages']; return it."""
    if role not in ('user', 'assistant', 'system'):
        raise ValueError(f'invalid role: {role!r}')
    msg = {'role': role, 'content': content}
    state['messages'].append(msg)
    return msg


def reset_messages(state: dict) -> None:
    """Clear the conversation but keep settings (a 'Clear chat' button)."""
    state['messages'] = []


def validate_user_input(text: str, max_chars: int = 2000) -> tuple[bool, str]:
    """
    Validate raw text from an st.chat_input / st.text_area widget before it is
    sent to the model.

    Returns (is_valid, result):
      - empty/whitespace : (False, 'Please enter a message.')
      - too long         : (False, 'Message too long (max N chars).')
      - valid            : (True, cleaned_text)   # stripped
    """
    cleaned = text.strip()
    if not cleaned:
        return (False, 'Please enter a message.')
    if len(cleaned) > max_chars:
        return (False, f'Message too long (max {max_chars} chars).')
    return (True, cleaned)


def clamp(value: float, lo: float, hi: float) -> float:
    """Clamp a widget value into [lo, hi]. st.slider bounds live input, but a
    value restored from session_state or a URL param may be out of range."""
    return max(lo, min(hi, value))


def build_settings(model: str, temperature: float, system_prompt: str) -> dict:
    """
    Assemble a validated settings dict from sidebar widget values.
    - temperature clamped to [0.0, 1.0]
    - system_prompt stripped; empty falls back to a default
    """
    sp = system_prompt.strip() or 'You are a helpful assistant.'
    return {
        'model': model,
        'temperature': float(clamp(temperature, 0.0, 1.0)),
        'system_prompt': sp,
    }


def build_messages(state: dict, user_text: str) -> list:
    """
    Build the messages list for ollama.chat:
        [system_prompt] + prior conversation + new user turn.
    Reads the system prompt from state['settings']. Does NOT mutate state.
    """
    settings = state.get('settings', {})
    system_prompt = settings.get('system_prompt', 'You are a helpful assistant.')
    messages = [{'role': 'system', 'content': system_prompt}]
    messages.extend(state.get('messages', []))
    messages.append({'role': 'user', 'content': user_text})
    return messages


def chat_with_history(state: dict, user_text: str, model: str = 'llama3.2') -> str:
    """
    Send the full conversation to Ollama and return the assistant's reply.
    Reads temperature from state['settings']. Returns a fallback string if
    Ollama is unavailable so the app never crashes on a model error.
    """
    settings = state.get('settings', {})
    temperature = settings.get('temperature', 0.7)
    messages = build_messages(state, user_text)
    try:
        response = ollama.chat(
            model=model,
            messages=messages,
            options={'temperature': temperature},
        )
        return response['message']['content'].strip()
    except Exception as e:
        return f'[Model unavailable: {e}]' 

## Your Implementation

In [ ]:
def format_transcript(messages: list) -> str:
    """
    Plain-text transcript for st.download_button. One block per turn as
    'ROLE: content'. Skip system messages. Join blocks with a blank line.
    """
    lines = []
    # TODO: for m in messages:
    #     role = m.get('role', '')
    #     if role == 'system': continue
    #     lines.append(f"{role.upper()}: {m.get('content', '')}")
    # TODO: return '\n\n'.join(lines)
    pass


def chat_stats(messages: list) -> dict:
    """
    Metrics for st.metric. System messages excluded.
    Return {'total', 'user', 'assistant', 'chars'}.
    """
    non_system = [m for m in messages if m.get('role') != 'system']
    # TODO: user = sum(1 for m in non_system if m.get('role') == 'user')
    # TODO: assistant = sum(1 for m in non_system if m.get('role') == 'assistant')
    # TODO: chars = sum(len(m.get('content', '')) for m in non_system)
    # TODO: return {'total': len(non_system), 'user': user, 'assistant': assistant, 'chars': chars}
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    convo = [
        {'role': 'system', 'content': 'You are helpful.'},
        {'role': 'user', 'content': 'hi'},
        {'role': 'assistant', 'content': 'hello there'},
        {'role': 'user', 'content': 'bye'},
    ]

    # Check 1: format_transcript returns a str with ROLE: prefixes
    try:
        t = format_transcript(convo)
        assert isinstance(t, str), f'expected str, got {type(t).__name__}'
        assert 'USER: hi' in t and 'ASSISTANT: hello there' in t, f'bad transcript:\n{t}'
        passed += 1; print('✅ Check 1: format_transcript uses ROLE: prefixes')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: system messages are skipped in the transcript
    try:
        t = format_transcript(convo)
        assert 'SYSTEM' not in t and 'You are helpful' not in t, 'system message leaked into transcript'
        passed += 1; print('✅ Check 2: system messages skipped')
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: chat_stats counts user + assistant correctly
    try:
        s = chat_stats(convo)
        assert s['user'] == 2, f"expected 2 user, got {s['user']}"
        assert s['assistant'] == 1, f"expected 1 assistant, got {s['assistant']}"
        passed += 1; print('✅ Check 3: chat_stats counts roles')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: total == user + assistant (system excluded)
    try:
        s = chat_stats(convo)
        assert s['total'] == 3, f"expected total 3 (no system), got {s['total']}"
        assert s['total'] == s['user'] + s['assistant'], 'total must equal user + assistant'
        passed += 1; print('✅ Check 4: total excludes system messages')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: chars counts non-system content characters
    try:
        s = chat_stats(convo)
        expected = len('hi') + len('hello there') + len('bye')
        assert s['chars'] == expected, f"expected {expected} chars, got {s['chars']}"
        passed += 1; print(f"✅ Check 5: chars={s['chars']} counts non-system content")
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def format_transcript(messages: list) -> str:
    """
    Render the conversation as a plain-text transcript for st.download_button.
    One block per turn as 'ROLE: content'. System messages are skipped.
    """
    lines = []
    for m in messages:
        role = m.get('role', '')
        if role == 'system':
            continue
        lines.append(f"{role.upper()}: {m.get('content', '')}")
    return '\n\n'.join(lines)


def chat_stats(messages: list) -> dict:
    """
    Compute display metrics for st.metric widgets. System messages excluded.
    Returns: {'total', 'user', 'assistant', 'chars'}.
    """
    non_system = [m for m in messages if m.get('role') != 'system']
    user = sum(1 for m in non_system if m.get('role') == 'user')
    assistant = sum(1 for m in non_system if m.get('role') == 'assistant')
    chars = sum(len(m.get('content', '')) for m in non_system)
    return {
        'total': len(non_system),
        'user': user,
        'assistant': assistant,
        'chars': chars,
    }
```

**Why this works:** Both functions are *pure* — same input, same output, no side effects — so they're trivial to test without a browser. `format_transcript` skips system messages because the user never sees them in the chat. `chat_stats` returns a flat dict that maps one-to-one onto `st.metric` calls, so the sidebar code is just `st.metric('Messages', chat_stats(msgs)['total'])`.
</details>